In [1]:
from collections import defaultdict
import json

import wandb


In [ ]:
from wandb.old.summary import SummarySubDict

api = wandb.Api()

def dictify(d):
    if isinstance(d, SummarySubDict):
        return {k: dictify(v) for k, v in d.items()}
    if isinstance(d, dict):
        return {k: dictify(v) for k, v in d.items()}
    elif isinstance(d, list):
        return [dictify(v) for v in d]
    else:
        return d

for ds in [
    "DeepPavlov/hwu64",
    "DeepPavlov/snips",
    "DeepPavlov/massive",
    "DeepPavlov/minds14",
]:
    runs = api.runs(
        "samoed-roman/new_autointent_encoders", 
        filters={
            "$and": [
                {"group": {"$regex": f"{ds}.*_\\d+"}},
                {"displayName": "final_metrics"},
            ]
        }
    )
    results = defaultdict(lambda: defaultdict(dict))
    for i, run in enumerate(runs):
        print(f"Run {i}: {run.name} - {run.group} - {run.config}")
        
        results[run.group] = {
            "config": dictify(run.config["scoring"][0]["module_params"]["embedder_config"]),
            "summary": {
                k: dictify(v)
                for k, v in run.summary.items() if k not in ["_wandb", "_timestamp", "_step"]
            },
        }
    with open(f"retrieval_{ds.replace('/', '__')}_different_retrievals.json", "w") as f:
        json.dump(results, f, indent=4)


Run 0: final_metrics - DeepPavlov/hwu64_intfloat/multilingual-e5-large_0 - {'regex': [], 'scoring': [{'metrics': {'scoring_roc_auc': 0.969148365244232}, 'metric_name': 'scoring_roc_auc', 'module_name': 'knn', 'metric_value': 0.969148365244232, 'module_params': {'k': 5, 'weights': 'distance', 'embedder_config': {'device': None, 'freeze': True, 'use_cache': True, 'batch_size': 32, 'model_name': 'intfloat/multilingual-e5-large', 'sts_prompt': None, 'query_prompt': None, 'cluster_prompt': None, 'default_prompt': None, 'passage_prompt': None, 'tokenizer_config': {'padding': True, 'max_length': None, 'truncation': True}, 'classifier_prompt': None, 'trust_remote_code': False, 'similarity_fn_name': 'cosine'}}, 'module_dump_dir': None}, {'metrics': {'scoring_roc_auc': 0.982855260673268}, 'metric_name': 'scoring_roc_auc', 'module_name': 'knn', 'metric_value': 0.982855260673268, 'module_params': {'k': 10, 'weights': 'distance', 'embedder_config': {'device': None, 'freeze': True, 'use_cache': True

In [45]:
flatten = defaultdict(dict)
for ds in [
    "DeepPavlov/hwu64",
    "DeepPavlov/snips",
    "DeepPavlov/massive",
    "DeepPavlov/minds14",
]:
    with open(f"retrieval_{ds.replace('/', '__')}_different_retrievals.json", "r") as f:
        results = json.load(f)
    for r in results:
        ds, model = r.split("_", 1)
        # print(f"Dataset: {ds}, Model: {model}")
        # print("Final metrics:", results[r]["summary"]["pipeline_metrics"])
        flatten[ds][model] = {
            "config": results[r]["config"],
            **{
                k: v for k, v in results[r]["summary"]["pipeline_metrics"].items()
            }
        }

In [ ]:
import pandas as pd

# Convert the nested dictionary to a DataFrame with multi-index
result_df = pd.concat({
    dataset: pd.DataFrame(models).T 
    for dataset, models in flatten.items()
})

In [50]:
result_df.to_csv("retrieval_different_retrievals.csv")

In [62]:
from wandb.old.summary import SummarySubDict

api = wandb.Api()

def dictify(d):
    if isinstance(d, SummarySubDict):
        return {k: dictify(v) for k, v in d.items()}
    if isinstance(d, dict):
        return {k: dictify(v) for k, v in d.items()}
    elif isinstance(d, list):
        return [dictify(v) for v in d]
    else:
        return d


results = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))
for ds in [
    "DeepPavlov/hwu64",
    "DeepPavlov/snips",
    "DeepPavlov/massive",
    "DeepPavlov/minds14",
]:
    runs = api.runs(
        f"samoed-roman/new_autointent_encoders", 
        filters={
            "$and": [
                {"group": ds},
                {
                    "$or": [
                        {"displayName": {"$regex": "retrieval_\\d+"}},
                        {"displayName": "final_metrics"},
                    ]
                }
            ]
        },
    )
    for i, run in enumerate(runs):
        print(f"Run {i}: {run.name} - {run.group} - {run.config}")
        
        results[run.group][run.name] = {
            "config": dictify(run.config),
            **{
                k: dictify(v)
                for k, v in run.summary.items() if k not in ["_wandb", "_timestamp", "_step"]
            },
        }
# with open(f"retrieval_{ds.replace('/', '__')}_different_retrievals.json", "w") as f:
#     json.dump(results, f, indent=4)


Run 0: retrieval_0 - DeepPavlov/hwu64 - {'k': 10, 'embedder_config': {'model_name': 'HIT-TMG/KaLM-embedding-multilingual-mini-instruct-v1.5', 'passage_prompt': 'Instruct: Create most accureate represantaion this text \n Query: ', 'trust_remote_code': True}}
Run 1: retrieval_1 - DeepPavlov/hwu64 - {'k': 10, 'embedder_config': {'model_name': 'microsoft/deberta-v3-large'}}
Run 2: retrieval_2 - DeepPavlov/hwu64 - {'k': 10, 'embedder_config': {'model_name': 'nomic-ai/nomic-embed-text-v1.5', 'query_prompt': 'search_query: ', 'passage_prompt': 'search_document: ', 'classifier_prompt': 'classification: ', 'trust_remote_code': True}}
Run 3: retrieval_3 - DeepPavlov/hwu64 - {'k': 10, 'embedder_config': {'model_name': 'WhereIsAI/UAE-Large-V1', 'trust_remote_code': True}}
Run 4: retrieval_4 - DeepPavlov/hwu64 - {'k': 10, 'embedder_config': {'model_name': 'intfloat/multilingual-e5-large', 'passage_prompt': 'query: '}}
Run 5: retrieval_5 - DeepPavlov/hwu64 - {'k': 10, 'embedder_config': {'model_name

In [63]:
result_df = pd.concat({
    dataset: pd.DataFrame(models).T 
    for dataset, models in results.items()
})
result_df

config  \
DeepPavlov/hwu64   retrieval_0    {'k': 10, 'embedder_config': {'model_name': 'H...   
                   retrieval_1    {'k': 10, 'embedder_config': {'model_name': 'm...   
                   retrieval_2    {'k': 10, 'embedder_config': {'model_name': 'n...   
                   retrieval_3    {'k': 10, 'embedder_config': {'model_name': 'W...   
                   retrieval_4    {'k': 10, 'embedder_config': {'model_name': 'i...   
...                                                                             ...   
DeepPavlov/minds14 retrieval_17   {'k': 10, 'embedder_config': {'model_name': 'a...   
                   retrieval_18   {'k': 10, 'embedder_config': {'model_name': 's...   
                   retrieval_19   {'k': 10, 'embedder_config': {'model_name': 'i...   
                   retrieval_20   {'k': 10, 'embedder_config': {'model_name': 'n...   
                   final_metrics  {'regex': [], 'scoring': [{'metrics': {'scorin...   

                                   _runtime retrieval_hit_rate retrieval_map  \
DeepPavlov/hwu64   retrieval_0    30.751424           0.969866      0.844418   
                   retrieval_1    28.279606            0.71875      0.449511   
                   retrieval_2    13.024112           0.946429      0.776048   
                   retrieval_3    19.697608           0.965402      0.873437   
                   retrieval_4    29.681838           0.966518      0.831045   
...                                     ...                ...           ...   
DeepPavlov/minds14 retrieval_17    4.229079           0.976744      0.959233   
                   retrieval_18    4.056931                  1      0.811751   
                   retrieval_19    0.818048                  1      0.940947   
                   retrieval_20    0.850044           0.976744      0.944474   
                   final_metrics    0.91868                NaN           NaN   

                                 retrieval_mrr retrieval_ndcg  \
DeepPavlov/hwu64   retrieval_0         0.87814       0.894532   
                   retrieval_1        0.505846       0.537076   
                   retrieval_2        0.826529       0.843296   
                   retrieval_3        0.900167       0.911237   
                   retrieval_4        0.879789        0.88673   
...                                        ...            ...   
DeepPavlov/minds14 retrieval_17       0.965116       0.967485   
                   retrieval_18       0.888372        0.88719   
                   retrieval_19        0.97093       0.967013   
                   retrieval_20       0.965116       0.963636   
                   final_metrics           NaN            NaN   

                                 retrieval_precision  \
DeepPavlov/hwu64   retrieval_0              0.776451   
                   retrieval_1              0.244196   
                   retrieval_2              0.668527   
                   retrieval_3              0.812723   
                   retrieval_4              0.728013   
...                                              ...   
DeepPavlov/minds14 retrieval_17             0.913953   
                   retrieval_18             0.544186   
                   retrieval_19             0.832558   
                   retrieval_20              0.82093   
                   final_metrics                 NaN   

                                                                            configs  \
DeepPavlov/hwu64   retrieval_0                                                  NaN   
                   retrieval_1                                                  NaN   
                   retrieval_2                                                  NaN   
                   retrieval_3                                                  NaN   
                   retrieval_4                                                  NaN   
...                                                                             ...   
DeepPavl

In [64]:
result_df.to_csv("retrieval_different_retrievals_full_config.csv")